# Streaming Signal Decomposition — Full Framework Demo

This notebook is a **complete, runnable walkthrough** of every public feature
of the Streaming SSD framework (DACS bachelor thesis, Maastricht University,
Spring 2026).  Executing it top-to-bottom verifies:

1. All four synthetic signal generators
2. Every decomposition engine: `SSD`, `SSA`, `IncrementalSSD`, `OptimizedSSD`,
   `RankOneIncrementalSSD`, plus `RankOneUpdater` and standalone `rsvd`
3. All similarity metrics (`d_corr`, `d_freq`, `w_correlation`, `subspace_angle`)
4. The full streaming pipeline (`WindowManager`, `ComponentMatcher`,
   `TrajectoryStore`), including stateless/stateful APIs, all distance modes,
   `max_cost` rejection, `max_trajectories` cap, and `lookback`
5. All stability metrics (`qrf`, `nmse`, `energy_continuity`,
   `singular_value_drift`, `dominant_frequency`, `freq_drift_aggregate`,
   `matching_confidence`)
6. Every visualization function in `src/visualization/`
7. Robustness to noise (QRF / NMSE vs SNR sweep)
8. Dynamic component onset detection
9. Integration with `experiments/run_experiment.py` via YAML configs

## 0. Setup — imports, seeds, output directory

In [28]:
import sys, os, json, time, yaml, warnings, importlib
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

# ---- signal generators ---------------------------------------------------
from experiments.synthetic.generators import (
    two_sinusoids, chirp_plus_sinusoid, rossler, component_onset,
)

# ---- decomposition engines -----------------------------------------------
# Force-reload src.engines so a stale kernel cache (loaded before
# ssd_optimized was registered) does not shadow the current __init__.py.
import src.engines as _eng_pkg
importlib.reload(_eng_pkg)

from src.engines import (
    DecompositionEngine, get_engine,
    SSD, SSA, IncrementalSSD, RankOneIncrementalSSD,
    RankOneUpdater,
    build_trajectory_matrix, svd_decompose, diagonal_averaging, auto_ssa,
    rsvd,
)
# Import OptimizedSSD and _REGISTRY directly from their source modules so
# they are always available regardless of __init__.py cache state.
from src.engines.ssd_optimized import OptimizedSSD
from src.engines import _REGISTRY
from src.engines.svd_update import _build_hankel

# ---- similarity metrics --------------------------------------------------
from src.metrics.similarity import d_corr, d_freq, w_correlation, subspace_angle

# ---- stability metrics ---------------------------------------------------
from src.metrics.stability import (
    qrf, nmse, energy_continuity, singular_value_drift,
    dominant_frequency, freq_drift_aggregate, matching_confidence,
)

# ---- streaming pipeline --------------------------------------------------
from src.streaming.window_manager import WindowManager
from src.streaming.component_matcher import ComponentMatcher
from src.streaming.trajectory_store import TrajectoryStore

# ---- visualization -------------------------------------------------------
from src.visualization import (
    plot_decomposition, plot_trajectory_overlay, plot_component_spectra,
    plot_matching_graph, plot_metrics_over_windows, plot_pipeline_dashboard,
    plot_window_reconstruction, plot_window_grid, plot_nmse_over_time,
)
from src.visualization.plot_metrics import plot_metrics

# ---- experiment runner ---------------------------------------------------
from experiments.run_experiment import build_pipeline, run as run_experiment

FS = 1000.0
SEED = 42
np.random.seed(SEED)

RESULTS_DIR = Path("../results/demo_run")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Results will be written to: {RESULTS_DIR.resolve()}")
print(f"OptimizedSSD available: {OptimizedSSD}")
print(f"Registered engines: {sorted(_REGISTRY.keys())}")

Results will be written to: /Users/sachaloeb/Documents/StreamingSignalDecompositionRTDA/results/demo_run
OptimizedSSD available: <class 'src.engines.ssd_optimized.OptimizedSSD'>
Registered engines: ['ssa', 'ssd', 'ssd_incremental', 'ssd_optimized', 'ssd_rank1']


## 1. Synthetic signal generators

We demonstrate all four generators from `experiments.synthetic.generators`.
`chirp_plus_sinusoid` is kept as the primary signal for later sections because
it is non-stationary and therefore exercises every part of the streaming
pipeline.

In [29]:
N = 3000

sig_two = two_sinusoids(N=N, f1=20.0, f2=80.0, fs=FS, snr_db=30.0, seed=SEED)
sig_chirp = chirp_plus_sinusoid(N=N, f_sin=50.0, f_start=10.0, f_end=150.0,
                                fs=FS, snr_db=20.0, seed=SEED)
sig_ross = rossler(N=N, dt=0.02, seed=SEED)
sig_onset = component_onset(N=N, f_steady=30.0, f_onset=90.0,
                            onset_sample=N // 2, fs=FS, seed=SEED)

assert sig_two.shape == sig_chirp.shape == sig_ross.shape == sig_onset.shape == (N,)

fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=False)
t = np.arange(N) / FS
for ax, sig, name in zip(
    axes,
    [sig_two, sig_chirp, sig_ross, sig_onset],
    ["two_sinusoids", "chirp_plus_sinusoid", "rossler", "component_onset"],
):
    ax.plot(t if name != "rossler" else np.arange(N), sig, linewidth=0.7)
    ax.set_title(name)
    ax.set_ylabel("amp")

# vertical line at onset for component_onset
axes[3].axvline(x=(N // 2) / FS, color="red", linestyle="--", label="onset")
axes[3].legend(fontsize=8)
axes[-1].set_xlabel("time (s) / samples")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "01_generators.png", dpi=120, bbox_inches="tight")
plt.show()

# primary signal for the rest of the notebook
signal = sig_chirp
print("Primary signal:", signal.shape, "energy =", round(float(np.dot(signal, signal)), 2))

Primary signal: (3000,) energy = 3052.62


## 2. Decomposition engines

### 2.1 `DecompositionEngine` base class + `get_engine` factory

`get_engine(name, fs, **kwargs)` dispatches by name from `_REGISTRY`.

In [30]:
eng_ssd = get_engine("ssd", fs=FS, nmse_threshold=0.01, max_iter=10)
eng_ssa = get_engine("ssa", fs=FS, n_components=3)
assert isinstance(eng_ssd, DecompositionEngine)
assert isinstance(eng_ssa, DecompositionEngine)
print("ssd ->", type(eng_ssd).__name__, "| ssa ->", type(eng_ssa).__name__)

# Show all registered engine names
print("Registered engines:", sorted(_REGISTRY.keys()))
assert set(_REGISTRY.keys()) == {"ssd", "ssa", "ssd_incremental", "ssd_optimized", "ssd_rank1"}

# Unknown engine should raise ValueError
try:
    get_engine("unknown", fs=FS)
except ValueError as exc:
    print("Correctly rejected unknown engine:", exc)

ssd -> SSD | ssa -> SSA
Registered engines: ['ssa', 'ssd', 'ssd_incremental', 'ssd_optimized', 'ssd_rank1']
Correctly rejected unknown engine: Unknown engine 'unknown'. Available: ['ssa', 'ssd', 'ssd_incremental', 'ssd_optimized', 'ssd_rank1']


### 2.2 SSD — full offline decomposition

The wrapped trajectory matrix, the automatically selected window length
`M ≈ 1.2 * fs / f_max`, and the iterative extraction with NMSE stopping
criterion.

In [31]:
ssd = SSD(fs=FS, nmse_threshold=0.02, max_iter=20)

# --- automatic window-length selection ---
M_auto = ssd._choose_window_length(signal)
print(f"_choose_window_length -> M = {M_auto}  (M ≈ 1.2 * fs / f_max)")

# --- wrapped trajectory matrix ---
X_wrapped = SSD._build_trajectory_matrix(signal, M_auto)
print("Wrapped trajectory matrix shape:", X_wrapped.shape)
assert X_wrapped.shape == (M_auto, len(signal))

# --- full decomposition ---
ssd_components = ssd.fit(signal)
ssd_residual = ssd_components[-1]
ssd_comps_only = ssd_components[:-1]
recon_ssd = np.sum(ssd_comps_only, axis=0)
ssd_nmse = nmse(signal - recon_ssd, signal)
print(f"SSD extracted {len(ssd_comps_only)} components (NMSE = {ssd_nmse:.5f})")

# Verify sum(components) + residual ≈ signal
assert np.allclose(recon_ssd + ssd_residual, signal, atol=1e-10)
print("sum(components) + residual == signal: True")

_choose_window_length -> M = 23  (M ≈ 1.2 * fs / f_max)
Wrapped trajectory matrix shape: (23, 3000)
SSD extracted 5 components (NMSE = 0.01725)
sum(components) + residual == signal: True


### 2.3 SSA primitives + `SSA` engine wrapper

In [32]:
sig_ssa = signal[:500]
L = 50

# build_trajectory_matrix (standard Hankel)
X_hankel = build_trajectory_matrix(sig_ssa, L=L)
print("Hankel trajectory matrix shape:", X_hankel.shape)
assert X_hankel.shape == (L, len(sig_ssa) - L + 1)

# svd_decompose
U, S, Vt = svd_decompose(X_hankel, rank=5)
print("Top-5 singular values:", np.round(S, 3))
assert U.shape == (L, 5) and Vt.shape == (5, len(sig_ssa) - L + 1)

# diagonal_averaging: reconstruct rank-1 approximation
X1 = S[0] * np.outer(U[:, 0], Vt[0, :])
recon_rank1 = diagonal_averaging(X1)
print("Rank-1 reconstruction length:", len(recon_rank1))
assert len(recon_rank1) == len(sig_ssa)

# auto_ssa hierarchical grouping
ssa_groups = auto_ssa(sig_ssa, r=3, L=L)
print(f"auto_ssa -> {len(ssa_groups)} groups, length {len(ssa_groups[0])}")

# SSA engine wrapper via get_engine
ssa_eng = get_engine("ssa", fs=FS, n_components=3, window_length=L)
ssa_eng_comps = ssa_eng.fit(sig_ssa)
print(f"SSA engine produced {len(ssa_eng_comps)} components")

Hankel trajectory matrix shape: (50, 451)
Top-5 singular values: [79.037 75.57  64.177 61.782 38.222]
Rank-1 reconstruction length: 500
auto_ssa -> 3 groups, length 500
SSA engine produced 3 components


### 2.4 `RankOneUpdater` + `RankOneIncrementalSSD`

Brand's (2003) rank-1 SVD update lets us modify a low-rank factorisation
`X ≈ U S Vᵀ` when `X` changes by an outer product `a bᵀ`.

In [33]:
# ── RankOneUpdater basics ──────────────────────────────────────────────
L_r1 = 50
X_h = _build_hankel(sig_ssa, L_r1)
U0, S0, Vt0 = svd_decompose(X_h, rank=4)
updater = RankOneUpdater(U0, S0, Vt0, refresh_every=100)
print("RankOneUpdater initialised:", updater.U.shape, updater.S.shape, updater.Vt.shape)

# Single rank-1 update: X_new = X + a @ b^T
rng = np.random.default_rng(7)
a = rng.standard_normal(L_r1) * 0.01
b = rng.standard_normal(X_h.shape[1]) * 0.01
U_new, S_new, Vt_new = updater.update(a, b)
X_exact = X_h + np.outer(a, b)
X_approx = U_new @ np.diag(S_new) @ Vt_new
rel_err = np.linalg.norm(X_exact - X_approx, 'fro') / np.linalg.norm(X_exact, 'fro')
print(f"Rank-1 update relative Frobenius error: {rel_err:.5f}")

# slide_window: 10 sequential slides
x_slide = sig_ssa.copy()
for extra in sig_ssa[len(sig_ssa)-10:]:
    updater.slide_window(float(extra), x_slide)
print(f"After 10 slide_window calls — singular values: {updater.S.round(2)}")

RankOneUpdater initialised: (50, 4) (4,) (4, 451)
Rank-1 update relative Frobenius error: 0.34030
After 10 slide_window calls — singular values: [89.42 69.15 62.11 56.88]


In [34]:
# ── RankOneIncrementalSSD: correctness + timing ──────────────────────
WLEN_R1, STRIDE_R1 = 500, 150
eng_baseline = SSD(fs=FS, nmse_threshold=0.02, max_iter=20)
eng_r1 = RankOneIncrementalSSD(fs=FS, stride=STRIDE_R1, rank=8,
                                refresh_every=50, nmse_threshold=0.01, max_iter=8)

demo_sig = chirp_plus_sinusoid(N=2000, f_sin=50, f_start=10, f_end=150,
                                fs=FS, snr_db=20, seed=SEED)

nmse_base, nmse_r1 = [], []
t_base, t_r1 = [], []

for eng, nlist, tlist in [(eng_baseline, nmse_base, t_base),
                           (eng_r1,      nmse_r1,   t_r1)]:
    wm_tmp = WindowManager(window_len=WLEN_R1, stride=STRIDE_R1, fs=FS)
    for samp in demo_sig:
        win = wm_tmp.push(float(samp))
        if win is None:
            continue
        t0 = time.perf_counter()
        c = eng.fit(win)
        tlist.append(time.perf_counter() - t0)
        recon = sum(c[:-1])
        nlist.append(nmse(win - recon, win))

print("NMSE comparison (median):")
print(f"  SSD baseline:        {np.median(nmse_base):.5f}")
print(f"  RankOneIncrementalSSD: {np.median(nmse_r1):.5f}")
print(f"\nTiming (warm, excluding cold start):")
print(f"  SSD baseline:  {1000*np.mean(t_base[1:]):.1f} ms / window")
print(f"  RankOne cold:  {1000*t_r1[0]:.1f} ms")
print(f"  RankOne warm:  {1000*np.mean(t_r1[1:]):.1f} ms / window")

NMSE comparison (median):
  SSD baseline:        0.01504
  RankOneIncrementalSSD: 0.00700

Timing (warm, excluding cold start):
  SSD baseline:  29.9 ms / window
  RankOne cold:  82.0 ms
  RankOne warm:  303.0 ms / window


### 2.5 `IncrementalSSD` (warm-start SVD caching)

Caches SVD factors from the previous window and reuses them when the subspace
has not drifted too much.  Falls back to a cold-start when it has.

In [35]:
# Cold start + warm start on two consecutive windows
eng_inc = get_engine("ssd_incremental", fs=FS, nmse_threshold=0.02, max_iter=20)
assert isinstance(eng_inc, IncrementalSSD)

w1 = signal[:500]
w2 = signal[150:650]

comps1 = eng_inc.fit(w1)
recon1 = np.sum(comps1[:-1], axis=0)
nmse1 = nmse(w1 - recon1, w1)
print(f"Cold start NMSE: {nmse1:.5f}")
assert nmse1 < 0.05

comps2 = eng_inc.fit(w2)
recon2 = np.sum(comps2[:-1], axis=0)
nmse2 = nmse(w2 - recon2, w2)
print(f"Warm start NMSE: {nmse2:.5f}")
assert nmse2 < 0.05

# rSVD variant
eng_inc_rsvd = IncrementalSSD(fs=FS, use_rsvd=True, nmse_threshold=0.02, max_iter=20)
comps_rsvd = eng_inc_rsvd.fit(w1)
recon_rsvd = np.sum(comps_rsvd[:-1], axis=0)
nmse_rsvd = nmse(w1 - recon_rsvd, w1)
print(f"rSVD variant NMSE: {nmse_rsvd:.5f}")
assert len(comps_rsvd) >= 2  # at least 1 component + residual

# Cold-start fallback: feed a drastically different signal
white_noise = np.random.default_rng(99).standard_normal(500) * 10.0
comps_noise = eng_inc.fit(white_noise)
assert len(comps_noise) >= 1
print(f"Cold-start fallback on noise: {len(comps_noise)-1} components (valid)")

Cold start NMSE: 0.01413
Warm start NMSE: 0.01927
rSVD variant NMSE: 0.01413
Cold-start fallback on noise: 6 components (valid)


### 2.6 `OptimizedSSD` (bandwidth estimation variants)

Inherits standard SSD and overrides bandwidth estimation with faster methods:
FWHM (O(N) walk), moment-based (second central moment), or Gaussian with
analytical Jacobian.

In [36]:
methods = ["fwhm", "moment", "gaussian"]
results_opt = {}

for method in methods:
    eng = OptimizedSSD(fs=FS, spectral_method=method, nmse_threshold=0.02, max_iter=20)
    t0 = time.perf_counter()
    comps = eng.fit(signal)
    elapsed = time.perf_counter() - t0
    recon = np.sum(comps[:-1], axis=0)
    n = nmse(signal - recon, signal)
    results_opt[method] = {"n_components": len(comps) - 1,
                           "nmse": round(n, 5),
                           "time_ms": round(1000 * elapsed, 1)}
    print(f"  {method:10s}  {len(comps)-1} comps  NMSE={n:.5f}  time={1000*elapsed:.1f} ms")

# Also via get_engine
eng_opt = get_engine("ssd_optimized", fs=FS, spectral_method="fwhm")
assert isinstance(eng_opt, OptimizedSSD)

# Compare against baseline SSD
t0 = time.perf_counter()
comps_base = SSD(fs=FS, nmse_threshold=0.02, max_iter=20).fit(signal)
t_base = 1000 * (time.perf_counter() - t0)
print(f"\n  baseline SSD  {len(comps_base)-1} comps  "
      f"NMSE={nmse(signal - np.sum(comps_base[:-1], axis=0), signal):.5f}  "
      f"time={t_base:.1f} ms")

  fwhm        3 comps  NMSE=0.01249  time=48.8 ms
  moment      5 comps  NMSE=0.01710  time=79.6 ms
  gaussian    5 comps  NMSE=0.01725  time=129.9 ms

  baseline SSD  5 comps  NMSE=0.01725  time=141.1 ms


### 2.7 `rsvd` standalone function

Randomised truncated SVD (Halko, Martinsson & Tropp, 2011).

In [37]:
X_traj = build_trajectory_matrix(signal[:500], L=50)
print("Trajectory matrix shape:", X_traj.shape)

# Full SVD
t0 = time.perf_counter()
U_full, S_full, Vt_full = np.linalg.svd(X_traj, full_matrices=False)
t_full = time.perf_counter() - t0

# rSVD
t0 = time.perf_counter()
U_r, S_r, Vt_r = rsvd(X_traj, k=5, n_oversamples=5, n_power_iter=1, seed=42)
t_rsvd = time.perf_counter() - t0

# Reconstruction error
X_recon_full = U_full[:, :5] @ np.diag(S_full[:5]) @ Vt_full[:5, :]
X_recon_rsvd = U_r @ np.diag(S_r) @ Vt_r
rel_err = np.linalg.norm(X_recon_full - X_recon_rsvd, 'fro') / np.linalg.norm(X_traj, 'fro')

print(f"Full SVD (top-5): {1e6*t_full:.0f} us")
print(f"rSVD (rank=5):    {1e6*t_rsvd:.0f} us")
print(f"Relative error (rSVD vs full SVD, rank-5): {rel_err:.6f}")
print(f"Singular values (full):  {S_full[:5].round(2)}")
print(f"Singular values (rSVD):  {S_r.round(2)}")

Trajectory matrix shape: (50, 451)
Full SVD (top-5): 1155 us
rSVD (rank=5):    462 us
Relative error (rSVD vs full SVD, rank-5): 0.000226
Singular values (full):  [79.04 75.57 64.18 61.78 38.22]
Singular values (rSVD):  [79.04 75.57 64.18 61.78 38.22]


## 3. Similarity metrics

Each metric demonstrated on pairs of SSD components.

In [38]:
g1 = ssd_comps_only[0]
g2 = ssd_comps_only[1] if len(ssd_comps_only) > 1 else ssd_comps_only[0]

print(f"d_corr(g1, g1) = {d_corr(g1, g1):.6f}   (self-distance ≈ 0)")
assert d_corr(g1, g1) < 1e-6
print(f"d_corr(g1, g2) = {d_corr(g1, g2):.4f}")
print(f"d_freq(g1, g2, fs=FS) = {d_freq(g1, g2, fs=FS):.2f} Hz")
print(f"w_correlation(g1, g2, L=200) = {w_correlation(g1, g2, L=200):.4f}")

# subspace_angle between two U-blocks from SSA of g1 and g2
U_a, _, _ = svd_decompose(build_trajectory_matrix(g1[:500], L=50), rank=2)
U_b, _, _ = svd_decompose(build_trajectory_matrix(g2[:500], L=50), rank=2)
theta = subspace_angle(U_a, U_b)
print(f"subspace_angle(U_a, U_b) = {theta:.4f} rad  ({np.degrees(theta):.1f} deg)")

d_corr(g1, g1) = -0.000000   (self-distance ≈ 0)
d_corr(g1, g2) = 0.9964
d_freq(g1, g2, fs=FS) = 78.12 Hz
w_correlation(g1, g2, L=200) = 0.0039
subspace_angle(U_a, U_b) = 1.5368 rad  (88.1 deg)


## 4. Streaming pipeline

### 4.1 `WindowManager`

In [39]:
WLEN, STRIDE = 300, 150
wm = WindowManager(window_len=WLEN, stride=STRIDE, fs=FS)
assert wm.overlap == WLEN - STRIDE == 150

emitted = []
for i, x in enumerate(signal):
    w = wm.push(float(x))
    if w is not None:
        emitted.append((i, w))

print(f"Emitted {len(emitted)} windows (first at sample {emitted[0][0]}, "
      f"last at sample {emitted[-1][0]})")
assert all(w.shape == (WLEN,) for _, w in emitted)

# Verify stride spacing
deltas = np.diff([i for i, _ in emitted])
assert np.all(deltas == STRIDE)
print("Stride spacing verified:", deltas[:5], "...")

wm.reset()
print("After reset: buffer cleared")

Emitted 19 windows (first at sample 299, last at sample 2999)
Stride spacing verified: [150 150 150 150 150] ...
After reset: buffer cleared


### 4.2 `ComponentMatcher`

In [40]:
# Prepare two consecutive windows and their SSD components
ssd_small = SSD(fs=FS, nmse_threshold=0.02, max_iter=20)
w_prev = signal[:WLEN]
w_curr = signal[STRIDE:STRIDE + WLEN]
comps_prev = ssd_small.fit(w_prev)[:-1]
comps_curr = ssd_small.fit(w_curr)[:-1]
overlap = WLEN - STRIDE
print(f"prev: {len(comps_prev)} comps | curr: {len(comps_curr)} comps | overlap: {overlap}")

# ── All three distance modes ──────────────────────────────────────────
for dist in ["d_corr", "d_freq", "hybrid"]:
    m = ComponentMatcher(distance=dist, freq_weight=0.5, fs=FS)
    cost = m.build_cost_matrix(comps_prev, comps_curr, overlap)
    mapping = m.match(comps_prev, comps_curr, overlap)
    print(f"  {dist:8s}  cost_shape={cost.shape}  mapping={mapping}")

# ── Stateful API ─────────────────────────────────────────────────────
matcher_sf = ComponentMatcher(distance="d_freq", freq_weight=1.0, fs=FS,
                              lookback=3, max_cost=0.6, max_trajectories=6)
m0 = matcher_sf.match_stateful(comps_prev, overlap)
m1 = matcher_sf.match_stateful(comps_curr, overlap)
prev_win_map = matcher_sf.previous_window_mapping()
print(f"\nStateful: window 0 -> {m0}")
print(f"Stateful: window 1 -> {m1}")
print(f"previous_window_mapping: {prev_win_map}")

prev: 3 comps | curr: 3 comps | overlap: 150
  d_corr    cost_shape=(3, 3)  mapping={0: 0, 1: 1, 2: 2}
  d_freq    cost_shape=(3, 3)  mapping={0: 0, 1: 1, 2: 2}
  hybrid    cost_shape=(3, 3)  mapping={0: 0, 1: 1, 2: 2}

Stateful: window 0 -> {0: 0, 1: 1, 2: 2}
Stateful: window 1 -> {0: 0, 1: 1, 2: 2}
previous_window_mapping: {0: 0, 1: 1, 2: 2}


In [41]:
# ── max_cost rejection ────────────────────────────────────────────────
# With a very tight threshold, most matches should be rejected (id = new)
matcher_tight = ComponentMatcher(distance="d_freq",freq_weight=1.0, fs=FS,
                                 max_cost=0.01, max_trajectories=20)
matcher_tight.match_stateful(comps_prev, overlap)
m_tight = matcher_tight.match_stateful(comps_curr, overlap)
print(f"max_cost=0.01 mapping: {m_tight}")
# All curr components should get new trajectory ids (not reusing prev ids)

# ── max_trajectories cap ─────────────────────────────────────────────
matcher_cap = ComponentMatcher(distance="d_freq",freq_weight=1.0, fs=FS,
                               max_cost=0.9, max_trajectories=2)
cap0 = matcher_cap.match_stateful(comps_prev, overlap)
print(f"max_trajectories=2, first window: {cap0}")
# Only first 2 components get real ids, rest get DROP_ID = -1
n_dropped = sum(1 for v in cap0.values() if v == -1)
print(f"  {n_dropped} components dropped (id=-1)")

# ── lookback parameter effect ────────────────────────────────────────
matcher_lb = ComponentMatcher(distance="d_freq",freq_weight=1.0, fs=FS,
                              lookback=5, max_cost=0.8, max_trajectories=10)
# Feed 5 windows to exercise the lookback buffer
ssd_lb = SSD(fs=FS, nmse_threshold=0.02, max_iter=20)
for win_i in range(5):
    start = win_i * STRIDE
    end = start + WLEN
    if end > len(signal):
        break
    win = signal[start:end]
    comps = ssd_lb.fit(win)[:-1]
    m = matcher_lb.match_stateful(comps, overlap)
    print(f"  lookback window {win_i}: mapping = {m}")

max_cost=0.01 mapping: {0: 0, 1: 1, 2: 3}
max_trajectories=2, first window: {0: 0, 1: 1, 2: -1}
  1 components dropped (id=-1)
  lookback window 0: mapping = {0: 0, 1: 1, 2: 2}
  lookback window 1: mapping = {0: 0, 1: 1, 2: 2}
  lookback window 2: mapping = {0: 0, 1: 1, 3: 2, 2: 3}
  lookback window 3: mapping = {0: 0, 1: 1, 2: 3, 3: 2, 4: 4}
  lookback window 4: mapping = {0: 0, 1: 1, 2: 3}


### 4.3 `TrajectoryStore`

In [42]:
store_demo = TrajectoryStore(max_components=10, max_len=len(signal))

# Fake two overlapping windows to exercise averaging
fake_a = np.ones(WLEN)
fake_b = 2.0 * np.ones(WLEN)
store_demo.update(0, [fake_a], {0: None}, overlap=0)
store_demo.update(STRIDE, [fake_b], {0: 0}, overlap=overlap)

traj0 = store_demo.get(0)
print("trajectory 0 length:", len(traj0))
avg_overlap = np.nanmean(traj0[STRIDE:WLEN])
print(f"Overlap region average (expect 1.5): {avg_overlap:.1f}")
assert abs(avg_overlap - 1.5) < 0.01

# get_all
all_trajs = store_demo.get_all()
print("get_all keys:", list(all_trajs.keys()))

# max_components cap enforcement
small = TrajectoryStore(max_components=1, max_len=500)
small.update(0, [np.zeros(WLEN), np.ones(WLEN)], {0: None, 1: None}, overlap=0)
assert len(small.get_all()) == 1
print("max_components cap enforced:", list(small.get_all().keys()))

trajectory 0 length: 450
Overlap region average (expect 1.5): 1.5
get_all keys: [0]
max_components cap enforced: [0]


### 4.4 Full streaming loop (end-to-end)

`WindowManager` → `SSD.fit` → `ComponentMatcher.match_stateful` →
`TrajectoryStore.update`, with all stability metrics computed per window.

In [43]:
MAX_COMPONENTS = 12
wm = WindowManager(window_len=WLEN, stride=STRIDE, fs=FS)
engine = SSD(fs=FS, nmse_threshold=0.02, max_iter=20)
matcher = ComponentMatcher(distance="d_freq", freq_weight=1.0, fs=FS,
                           lookback=15, max_cost=0.10, max_trajectories=MAX_COMPONENTS)

store = TrajectoryStore(max_components=MAX_COMPONENTS, max_len=len(signal))

overlap = wm.overlap
prev_components = None
prev_S = None
metrics_rows = []
pipeline_records = []
window_idx = 0

for sample_idx, x in enumerate(signal):
    window = wm.push(float(x))
    if window is None:
        continue

    components = engine.fit(window)
    comps = components[:-1]

    mapping = dict(matcher.match_stateful(comps, overlap))
    prev_win_map = matcher.previous_window_mapping()

    window_start = sample_idx - WLEN + 1
    store.update(window_start, comps, mapping, overlap)

    recon = np.sum(comps, axis=0) if comps else np.zeros_like(window)
    q_val = qrf(window, recon)

    Lw = max(2, len(window) // 3)
    Xw = build_trajectory_matrix(window, Lw)
    _, S_curr, _ = svd_decompose(Xw)
    sv_drift = singular_value_drift(S_curr, prev_S)
    prev_S = S_curr

    ec = energy_continuity(comps, prev_components, prev_win_map)

    if prev_components is not None and prev_win_map:
        C = matcher.build_cost_matrix(prev_components, comps, overlap)
        mc = matching_confidence(C, prev_win_map)
    else:
        mc = float("nan")

    row = {"window_index": window_idx, "qrf": q_val,
           "singular_value_drift": sv_drift,
           "energy_continuity": ec, "matching_confidence": mc}
    for ci, comp in enumerate(comps):
        row[f"f_max_c{ci}"] = dominant_frequency(comp, fs=FS)
    for ci in range(len(comps), MAX_COMPONENTS):
        row[f"f_max_c{ci}"] = np.nan
    metrics_rows.append(row)

    pipeline_records.append({
        "window_idx": window_idx, "sample_start": window_start,
        "window_signal": window.copy(),
        "components": [c.copy() for c in comps],
    })

    prev_components = comps
    window_idx += 1

metrics_df = pd.DataFrame(metrics_rows)
metrics_csv_path = str(RESULTS_DIR / "metrics.csv")
metrics_df.to_csv(metrics_csv_path, index=False)
print(f"Processed {window_idx} windows. Metrics saved to {metrics_csv_path}")
print(metrics_df[["window_index", "qrf", "singular_value_drift",
                   "matching_confidence"]].head())

Processed 19 windows. Metrics saved to ../results/demo_run/metrics.csv
   window_index        qrf  singular_value_drift  matching_confidence
0             0  18.503211                   NaN                  NaN
1             1  17.902634              3.747692             0.991111
2             2  17.630775              3.784173             0.991111
3             3  19.015219              5.683492             0.993333
4             4  20.496243             39.196116             0.995556


In [44]:
# ── Global reconstruction from trajectory store ───────────────────────
trajs = store.get_all()
full_recon = np.zeros(len(signal), dtype=np.float64)
coverage = np.zeros(len(signal), dtype=np.float64)
for arr in trajs.values():
    k = min(len(arr), len(signal))
    valid = ~np.isnan(arr[:k])
    full_recon[:k][valid] += arr[:k][valid]
    coverage[:k][valid] += 1.0

covered = coverage > 0
full_recon_masked = np.where(covered, full_recon, np.nan)

# Global QRF and NMSE over covered samples only
sig_cov = signal[covered]
rec_cov = full_recon[covered]
global_qrf  = qrf(sig_cov, rec_cov)
global_nmse = nmse(sig_cov - rec_cov, sig_cov)

print(f"Global QRF  : {global_qrf:.2f} dB")
print(f"Global NMSE : {global_nmse:.6f}")
print(f"Coverage    : {covered.sum()}/{len(signal)} samples ({100*covered.mean():.1f}%)")

# ── Plot ──────────────────────────────────────────────────────────────
t_full = np.arange(len(signal)) / FS
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

axes[0].plot(t_full, signal, linewidth=0.7, color="steelblue", label="original")
axes[0].plot(t_full, full_recon_masked, linewidth=0.7, color="darkorange",
             alpha=0.85, label="reconstruction")
axes[0].set_ylabel("amplitude")
axes[0].set_title(
    f"Streaming reconstruction vs original  |  "
    f"QRF = {global_qrf:.1f} dB,  NMSE = {global_nmse:.5f}"
)
axes[0].legend(fontsize=8)

residual_full = signal - np.where(covered, full_recon, signal)
axes[1].plot(t_full, residual_full, linewidth=0.7, color="crimson", label="residual")
axes[1].set_ylabel("amplitude")
axes[1].set_xlabel("time (s)")
axes[1].set_title("Residual  (original − reconstruction)")
axes[1].legend(fontsize=8)

fig.tight_layout()
fig.savefig(RESULTS_DIR / "04_global_reconstruction.png", dpi=120, bbox_inches="tight")
plt.show()

Global QRF  : 19.47 dB
Global NMSE : 0.011297
Coverage    : 3000/3000 samples (100.0%)


## 5. Stability metrics

Every function from `src.metrics.stability` — computed in the loop above,
aggregated here. Also shows the global post-hoc `freq_drift_aggregate`.

In [45]:
# Per-window QRF plot
fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex=True)
axes = axes.ravel()
for ax, col, color in zip(axes,
    ["qrf", "singular_value_drift", "energy_continuity", "matching_confidence"],
    ["blue", "orange", "green", "red"]):
    vals = metrics_df[col].values
    finite = np.isfinite(vals)
    ax.plot(metrics_df["window_index"].values[finite], vals[finite],
            color=color, linewidth=1)
    ax.set_title(col); ax.set_ylabel(col)
axes[2].set_xlabel("window index"); axes[3].set_xlabel("window index")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "05_stability_metrics.png", dpi=120, bbox_inches="tight")
plt.show()

# Dominant frequency per component
for ci in range(min(3, len(ssd_comps_only))):
    f = dominant_frequency(ssd_comps_only[ci], fs=FS)
    print(f"Component {ci} dominant freq: {f:.2f} Hz")

# Post-hoc freq_drift_aggregate — sort numerically to avoid c10 < c2 lexicographic trap
fmax_cols = sorted(
    [c for c in metrics_df.columns if c.startswith("f_max_c")],
    key=lambda x: int(x.split("f_max_c")[1]),
)
for c in fmax_cols[:4]:
    fd = freq_drift_aggregate(metrics_df[c].values)
    print(f"freq_drift_aggregate({c}) = {fd:.4f}" if np.isfinite(fd) else f"freq_drift_aggregate({c}) = NaN")

# Summary table
summary = {
    "mean_qrf_dB": float(metrics_df["qrf"].replace([np.inf], np.nan).mean()),
    "mean_sv_drift": float(metrics_df["singular_value_drift"].mean(skipna=True)),
    "mean_energy_cont": float(metrics_df["energy_continuity"].mean(skipna=True)),
    "mean_match_conf": float(metrics_df["matching_confidence"].mean(skipna=True)),
}
summary_df = pd.DataFrame([summary])
print("\nAggregated stability metrics:")
print(summary_df.to_string(index=False))

# Save run_summary.json for plot_metrics
# Keys use "t" prefix to match the freq_drift_t\d+ pattern expected by plot_metrics.py
run_summary = {**summary}
for i, c in enumerate(fmax_cols):
    v = freq_drift_aggregate(metrics_df[c].values)
    run_summary[f"freq_drift_t{i}"] = None if not np.isfinite(v) else v
with open(RESULTS_DIR / "run_summary.json", "w") as fh:
    json.dump(run_summary, fh, indent=2)

Component 0 dominant freq: 50.78 Hz
Component 1 dominant freq: 128.91 Hz
Component 2 dominant freq: 15.62 Hz
freq_drift_aggregate(f_max_c0) = 3.0433
freq_drift_aggregate(f_max_c1) = 1686.8359
freq_drift_aggregate(f_max_c2) = 587.3077
freq_drift_aggregate(f_max_c3) = 3.8147

Aggregated stability metrics:
 mean_qrf_dB  mean_sv_drift  mean_energy_cont  mean_match_conf
   18.676793      10.508822      10724.569533         0.995679


## 6. Visualizations

Every plotting function in `src/visualization/`, saving each to `RESULTS_DIR`.

In [46]:
# 6.1 plot_decomposition
plot_decomposition(signal, ssd_comps_only, residual=ssd_residual, fs=FS,
                   title="SSD Decomposition (offline)",
                   save_path=str(RESULTS_DIR / "06_decomposition.png"))

# 6.2 plot_trajectory_overlay
plot_trajectory_overlay(store, signal, fs=FS,
                        save_path=str(RESULTS_DIR / "06_trajectory_overlay.png"))

# 6.3 plot_component_spectra
plot_component_spectra(ssd_comps_only, fs=FS, nperseg=256,
                       save_path=str(RESULTS_DIR / "06_component_spectra.png"))

# 6.4 plot_matching_graph
ms = ComponentMatcher(distance="d_freq", freq_weight=1.0, fs=FS)
C_vis = ms.build_cost_matrix(comps_prev, comps_curr, overlap=WLEN - STRIDE)
map_vis = ms.match(comps_prev, comps_curr, overlap=WLEN - STRIDE)
plot_matching_graph(comps_prev, comps_curr, map_vis,
                    overlap=WLEN - STRIDE, cost_matrix=C_vis,
                    save_path=str(RESULTS_DIR / "06_matching_graph.png"))

print("Saved: decomposition, trajectory overlay, component spectra, matching graph")

Saved: decomposition, trajectory overlay, component spectra, matching graph


In [47]:
# 6.5 plot_metrics_over_windows
plot_metrics_over_windows(metrics_csv_path,
                          save_path=str(RESULTS_DIR / "06_metrics_over_windows.png"))

# 6.6 plot_pipeline_dashboard
plot_pipeline_dashboard(signal, ssd_comps_only, store, metrics_csv_path, fs=FS,
                        save_path=str(RESULTS_DIR / "06_pipeline_dashboard.png"))

# 6.7 plot_window_reconstruction — first, middle, last
n_rec = len(pipeline_records)
for label, idx in [("first", 0), ("middle", n_rec // 2), ("last", n_rec - 1)]:
    rec = pipeline_records[idx]
    plot_window_reconstruction(rec["window_signal"], rec["components"],
                               rec["window_idx"], rec["sample_start"], fs=FS,
                               save_path=str(RESULTS_DIR / f"06_window_{label}.png"))

# 6.8 plot_window_grid
plot_window_grid(pipeline_records, n_windows=9, fs=FS,
                 save_path=str(RESULTS_DIR / "06_window_grid.png"))

print("Saved: metrics_over_windows, pipeline_dashboard, window reconstructions, window_grid")

Saved: metrics_over_windows, pipeline_dashboard, window reconstructions, window_grid


In [48]:
# 6.9 plot_nmse_over_time
trajs = store.get_all()
full_recon = np.zeros_like(signal, dtype=np.float64)
coverage = np.zeros(len(signal), dtype=np.float64)
for arr in trajs.values():
    k = min(len(arr), len(full_recon))
    valid = ~np.isnan(arr[:k])
    full_recon[:k][valid] += arr[:k][valid]
    coverage[:k][valid] += 1.0
full_recon_plot = full_recon.copy()
full_recon_plot[coverage == 0] = np.nan

plot_nmse_over_time(signal, full_recon_plot, fs=FS,
                    save_path=str(RESULTS_DIR / "06_nmse_over_time.png"))

# 6.10 plot_metrics (advanced 3-panel from plot_metrics.py)
plot_metrics(RESULTS_DIR, show=False)

print("Saved: nmse_over_time, metrics_plot (PDF+PNG)")
print("All 10 visualization functions called and saved.")

Saved: nmse_over_time, metrics_plot (PDF+PNG)
All 10 visualization functions called and saved.


## 7. Robustness — QRF / NMSE vs SNR sweep

Decompose the primary signal under increasing noise levels and track
reconstruction quality.

In [49]:
snr_values = [5, 10, 15, 20, 30, 40, np.inf]
snr_rows = []

for snr in snr_values:
    noisy = chirp_plus_sinusoid(N=N, f_sin=50.0, f_start=10.0, f_end=150.0,
                                fs=FS,
                                snr_db=snr if np.isfinite(snr) else None,
                                seed=SEED)
    comps = SSD(fs=FS, nmse_threshold=0.01, max_iter=8).fit(noisy)
    recon = np.sum(comps[:-1], axis=0)
    snr_rows.append({
        "snr_db": snr if np.isfinite(snr) else "inf",
        "qrf_db": round(qrf(noisy, recon), 2),
        "nmse": round(nmse(noisy - recon, noisy), 5),
        "n_components": len(comps) - 1,
    })

snr_df = pd.DataFrame(snr_rows)
print(snr_df.to_string(index=False))

# Twin-axis plot
fig, ax1 = plt.subplots(figsize=(8, 4))
x_pos = np.arange(len(snr_values))
labels = [str(s) if np.isfinite(s) else "inf" for s in snr_values]

color_q = "steelblue"
ax1.plot(x_pos, snr_df["qrf_db"].values, "o-", color=color_q, label="QRF (dB)")
ax1.set_xlabel("SNR (dB)")
ax1.set_ylabel("QRF (dB)", color=color_q)
ax1.tick_params(axis="y", labelcolor=color_q)
ax1.set_xticks(x_pos)
ax1.set_xticklabels(labels)

ax2 = ax1.twinx()
color_n = "darkorange"
ax2.plot(x_pos, snr_df["nmse"].values, "s--", color=color_n, label="NMSE")
ax2.set_ylabel("NMSE", color=color_n)
ax2.tick_params(axis="y", labelcolor=color_n)

fig.suptitle("QRF and NMSE vs SNR")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "07_snr_sweep.png", dpi=120, bbox_inches="tight")
plt.show()

snr_db  qrf_db    nmse  n_components
     5   17.63 0.01725             8
    10   19.97 0.01006             8
    15   20.86 0.00820             8
    20   20.77 0.00838             7
    30   21.60 0.00692             6
    40   21.97 0.00635             6
   inf   20.95 0.00803             5


## 8. Dynamic component onset detection

Using `component_onset`, we run the streaming pipeline and track the number
of active trajectories over time.

In [50]:
onset_N = 4000
onset_sig = component_onset(N=onset_N, f_steady=30.0, f_onset=120.0,
                            onset_sample=2000, fs=FS, seed=SEED)

wm2 = WindowManager(window_len=400, stride=200, fs=FS)
eng2 = SSD(fs=FS, nmse_threshold=0.02, max_iter=6)
match2 = ComponentMatcher(distance="d_freq", freq_weight=1.0, fs=FS,
                          lookback=10, max_cost=0.10, max_trajectories=4)
store2 = TrajectoryStore(max_components=4, max_len=onset_N)

active_counts = []
for i, x in enumerate(onset_sig):
    w = wm2.push(float(x))
    if w is None:
        continue
    comps = eng2.fit(w)[:-1]
    mapping = dict(match2.match_stateful(comps, wm2.overlap))
    store2.update(i - wm2.window_len + 1, comps, mapping, wm2.overlap)
    active_counts.append((i / FS, len(set(mapping.values()) - {-1})))

tt, cc = zip(*active_counts)
fig, ax = plt.subplots(figsize=(10, 3))
ax.step(tt, cc, where="post", color="purple")
ax.axvline(2000 / FS, color="red", linestyle="--", label="onset at t=2.0 s")
ax.set_xlabel("time (s)")
ax.set_ylabel("# active components")
ax.set_title("Component onset detection")
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "08_onset_detection.png", dpi=120, bbox_inches="tight")
plt.show()

# Verify TrajectoryStore reuses freed IDs
keys = sorted(store2.get_all().keys())
print(f"Trajectory store keys: {keys}")
print(f"Total trajectories: {len(keys)} (capped by max_components=4)")

Trajectory store keys: [0, 1]
Total trajectories: 2 (capped by max_components=4)


## 9. Experiment runner integration

`experiments/run_experiment.py` ties everything together via a YAML config.

In [51]:
with open("../experiments/configs/baseline.yaml") as fh:
    baseline_cfg = yaml.safe_load(fh)
print("baseline.yaml:")
print(yaml.dump(baseline_cfg, sort_keys=False))

baseline.yaml:
signal:
  type: chirp_plus_sinusoid
  N: 3000
  fs: 1000.0
  f_sin: 50.0
  f_start: 10.0
  f_end: 150.0
  snr_db: 20.0
streaming:
  window_len: 300
  stride: 150
  max_components: 100
engine:
  name: ssd
  nmse_threshold: 0.02
  max_iter: 20
matcher:
  distance: d_freq
  freq_weight: 1.0
  lookback: 10
  max_cost: 0.1
output:
  save_trajectories: true
  save_metrics: true



In [52]:
# build_pipeline
cfg_copy = yaml.safe_load(yaml.dump(baseline_cfg))
wm_bp, eng_bp, match_bp, store_bp = build_pipeline(cfg_copy,
                                                   signal_length=baseline_cfg["signal"]["N"])
print("build_pipeline produced:",
      type(wm_bp).__name__, type(eng_bp).__name__,
      type(match_bp).__name__, type(store_bp).__name__)
assert isinstance(wm_bp, WindowManager)
assert isinstance(eng_bp, DecompositionEngine)
assert isinstance(match_bp, ComponentMatcher)
assert isinstance(store_bp, TrajectoryStore)

build_pipeline produced: WindowManager SSD ComponentMatcher TrajectoryStore


In [53]:
# Full end-to-end experiment
exp_cfg = yaml.safe_load(yaml.dump(baseline_cfg))
exp_out = str(RESULTS_DIR / "experiment_run")
run_experiment(config_dict=exp_cfg, output_dir=exp_out)

exp_metrics = pd.read_csv(Path(exp_out) / "metrics.csv")
with open(Path(exp_out) / "run_summary.json") as fh:
    exp_summary = json.load(fh)

print(f"\nmetrics.csv: {exp_metrics.shape[0]} windows")
print(exp_metrics[["window_index", "qrf", "matching_confidence"]].head())
print(f"\nrun_summary.json (first 5 entries):")
for k, v in list(exp_summary.items())[:5]:
    print(f"  {k}: {v}")

Streaming: 100%|██████████| 3000/3000 [00:00<00:00, 4709.66it/s]


metrics.csv: 19 windows
   window_index        qrf  matching_confidence
0             0  18.503211                  NaN
1             1  17.902634             0.991111
2             2  17.630775             0.991111
3             3  19.015219             0.993333
4             4  20.496243             0.995556

run_summary.json (first 5 entries):
  freq_drift_t0: 1014.0120487794323
  freq_drift_t1: 169.4105998961219
  freq_drift_t2: 56.26678466796875
  freq_drift_t3: 356.4453125
  freq_drift_t4: None


## 10. Coverage summary

| Module | Symbols demonstrated | Section |
|---|---|---|
| `experiments/synthetic/generators.py` | `two_sinusoids`, `chirp_plus_sinusoid`, `rossler`, `component_onset` | 1, 8 |
| `src/engines/base.py` | `DecompositionEngine` | 2.1 |
| `src/engines/__init__.py` | `get_engine`, `_REGISTRY` | 2.1 |
| `src/engines/ssd.py` | `SSD.fit`, `_build_trajectory_matrix`, `_choose_window_length` | 2.2 |
| `src/engines/ssa.py` | `build_trajectory_matrix`, `svd_decompose`, `diagonal_averaging`, `auto_ssa`, `SSA` | 2.3 |
| `src/engines/svd_update.py` | `RankOneUpdater` (`.update`, `.slide_window`), `_build_hankel` | 2.4 |
| `src/engines/ssd_rank1.py` | `RankOneIncrementalSSD` | 2.4 |
| `src/engines/ssd_incremental.py` | `IncrementalSSD` (cold/warm/rSVD/fallback) | 2.5 |
| `src/engines/ssd_optimized.py` | `OptimizedSSD` (`fwhm`, `moment`, `gaussian`) | 2.6 |
| `src/engines/rsvd.py` | `rsvd` | 2.7 |
| `src/metrics/similarity.py` | `d_corr`, `d_freq`, `w_correlation`, `subspace_angle` | 3 |
| `src/streaming/window_manager.py` | `WindowManager` (`.push`, `.overlap`, `.reset`) | 4.1 |
| `src/streaming/component_matcher.py` | `ComponentMatcher` (`.match`, `.build_cost_matrix`, `.match_stateful`, `.previous_window_mapping`), all 3 distance modes, `max_cost`, `max_trajectories`, `lookback` | 4.2 |
| `src/streaming/trajectory_store.py` | `TrajectoryStore` (`.update`, `.get`, `.get_all`, `max_components`) | 4.3 |
| (full loop) | End-to-end streaming pipeline | 4.4 |
| `src/metrics/stability.py` | `qrf`, `nmse`, `energy_continuity`, `singular_value_drift`, `dominant_frequency`, `freq_drift_aggregate`, `matching_confidence` | 4.4, 5 |
| `src/visualization/component_plots.py` | `plot_decomposition`, `plot_trajectory_overlay`, `plot_component_spectra`, `plot_matching_graph` | 6 |
| `src/visualization/metrics_plots.py` | `plot_metrics_over_windows` | 6 |
| `src/visualization/pipeline_dashboard.py` | `plot_pipeline_dashboard` | 6 |
| `src/visualization/window_inspector.py` | `plot_window_reconstruction`, `plot_window_grid`, `plot_nmse_over_time` | 6 |
| `src/visualization/plot_metrics.py` | `plot_metrics` | 6 |
| (robustness) | QRF / NMSE vs SNR sweep | 7 |
| (dynamics) | Component onset detection + reusable-id resolver | 8 |
| `experiments/run_experiment.py` | `build_pipeline`, `run` | 9 |

In [54]:
print("All framework features demonstrated. End of notebook.")

All framework features demonstrated. End of notebook.
